# Final Task B 17-class - ConvNeXt-Tiny

Train ConvNeXt-Tiny directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [1]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "final-B"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"final-B: {FINAL_B_DIR}")


ROOT: /workspace/woodbench
final-B: /workspace/woodbench/notebooks/phase 1/final-B


In [2]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    %pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [3]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "convnext_tiny"
DISPLAY_NAME = "ConvNeXt-Tiny"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: ConvNeXt-Tiny (convnext_tiny)
Batch size: 32
Data root: /workspace/woodbench/data/task-B-group-split


In [4]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-17class

main/seed_42/convnext_tiny
[main | convnext_tiny s42] train=3365, val=903, test=532, classes=17


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 3.2394, Acc: 2.74% | Val Loss: 3.2043, Acc: 5.20% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 5.20%)


Epoch 2/50 | Train Loss: 1.1090, Acc: 82.86% | Val Loss: 0.7257, Acc: 95.79% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 95.79%)


Epoch 3/50 | Train Loss: 0.6666, Acc: 98.18% | Val Loss: 0.6835, Acc: 96.46% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 96.46%)


Epoch 4/50 | Train Loss: 0.6356, Acc: 98.63% | Val Loss: 0.6315, Acc: 98.78% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 98.78%)


... (93 outputs left out)

Epoch 18/50 | Train Loss: 0.5735, Acc: 100.00% | Val Loss: 0.5825, Acc: 99.67% | LR: 8.08e-05[EarlyStop 7/10]


Epoch 19/50 | Train Loss: 0.5734, Acc: 100.00% | Val Loss: 0.5811, Acc: 99.67% | LR: 7.80e-05[EarlyStop 8/10]


Epoch 20/50 | Train Loss: 0.5734, Acc: 100.00% | Val Loss: 0.5804, Acc: 99.67% | LR: 7.50e-05[EarlyStop 9/10]


Epoch 21/50 | Train Loss: 0.5733, Acc: 100.00% | Val Loss: 0.5820, Acc: 99.67% | LR: 7.19e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 21.


In [ ]:
summarize_results(results)


## Continue after Vast interruption

Run the root, import/config, and data-check cells above, but do not rerun the old `run_all_experiments` cell. Then run each config cell below independently. Completed metrics are reused; interrupted checkpoints restore model, optimizer, and scheduler state.

In [5]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [6]:
from final_b_17class import EXPERIMENTS, load_completed_results, train_one

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 4/6 completed configs for convnext_tiny.


In [7]:
# Resume label-smoothing=0.0 from its saved checkpoint
run_remaining("taskB_label_smoothing_0.0", 42)


[taskB_label_smoothing_0.0 | convnext_tiny s42] train=3365, val=903, test=532, classes=17
[Resume] /workspace/woodbench/data/experiments/Task-B-17class/taskB_label_smoothing_0.0/convnext_tiny/convnext_tiny_seed42_best.pth -> epoch 11/50


Epoch 11/50 | Train Loss: 0.0031, Acc: 99.91% | Val Loss: 0.0173, Acc: 99.45% | LR: 9.57e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 99.45%)


Epoch 12/50 | Train Loss: 0.0131, Acc: 99.46% | Val Loss: 0.1052, Acc: 96.23% | LR: 9.41e-05[EarlyStop 1/10]


Epoch 13/50 | Train Loss: 0.0470, Acc: 98.48% | Val Loss: 0.1195, Acc: 96.79% | LR: 9.24e-05[EarlyStop 2/10]


Epoch 14/50 | Train Loss: 0.0396, Acc: 98.75% | Val Loss: 0.0352, Acc: 98.89% | LR: 9.05e-05[EarlyStop 3/10]


Epoch 15/50 | Train Loss: 0.0131, Acc: 99.64% | Val Loss: 0.0739, Acc: 97.34% | LR: 8.83e-05[EarlyStop 4/10]


... (10 outputs left out)

Epoch 26/50 | Train Loss: 0.0001, Acc: 100.00% | Val Loss: 0.0112, Acc: 99.56% | LR: 5.52e-05[EarlyStop 8/10]


Epoch 27/50 | Train Loss: 0.0001, Acc: 100.00% | Val Loss: 0.0110, Acc: 99.56% | LR: 5.17e-05[EarlyStop 9/10]


Epoch 28/50 | Train Loss: 0.0001, Acc: 100.00% | Val Loss: 0.0102, Acc: 99.56% | LR: 4.83e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 28.
[taskB_label_smoothing_0.0 | convnext_tiny s42] val_acc=99.67% test_acc=99.62% F1=99.66%


{'backbone': 'convnext_tiny',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.0',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 99.66777408637874,
 'best_epoch': 18,
 'final_train_acc': 100.0,
 'final_val_acc': 99.55703211517165,
 'stopped_early': True,
 'epochs_trained': 28,
 'test_accuracy': 0.9962406015037594,
 'test_f1_macro': 0.9965600586703852,
 'test_precision_macro': 0.9964873649994758,
 'test_recall_macro': 0.9967320261437908,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecyparis obtus

In [8]:
# Remaining label-smoothing=0.2 ablation
run_remaining("taskB_label_smoothing_0.2", 42)


[taskB_label_smoothing_0.2 | convnext_tiny s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 3.2362, Acc: 2.74% | Val Loss: 3.2047, Acc: 5.20% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 5.20%)


Epoch 2/50 | Train Loss: 1.4855, Acc: 82.08% | Val Loss: 1.1516, Acc: 96.23% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 96.23%)


Epoch 3/50 | Train Loss: 1.0930, Acc: 98.30% | Val Loss: 1.1077, Acc: 96.90% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 96.90%)


Epoch 4/50 | Train Loss: 1.0580, Acc: 98.99% | Val Loss: 1.0662, Acc: 98.34% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to convnext_tiny_seed42_best.pth (Val Acc: 98.34%)


Epoch 5/50 | Train Loss: 1.0560, Acc: 98.75% | Val Loss: 1.1050, Acc: 97.12% | LR: 1.00e-04[EarlyStop 1/10]


... (10 outputs left out)

Epoch 16/50 | Train Loss: 1.0408, Acc: 98.69% | Val Loss: 1.0416, Acc: 99.22% | LR: 8.60e-05[EarlyStop 8/10]


Epoch 17/50 | Train Loss: 1.0382, Acc: 98.72% | Val Loss: 1.1058, Acc: 95.57% | LR: 8.35e-05[EarlyStop 9/10]


Epoch 18/50 | Train Loss: 1.0381, Acc: 98.69% | Val Loss: 1.0417, Acc: 98.45% | LR: 8.08e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 18.
[taskB_label_smoothing_0.2 | convnext_tiny s42] val_acc=99.45% test_acc=99.06% F1=99.10%


{'backbone': 'convnext_tiny',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 99.44629014396456,
 'best_epoch': 8,
 'final_train_acc': 98.69047619047619,
 'final_val_acc': 98.44961240310077,
 'stopped_early': True,
 'epochs_trained': 18,
 'test_accuracy': 0.9906015037593985,
 'test_f1_macro': 0.9910272561289053,
 'test_precision_macro': 0.9910840673167662,
 'test_recall_macro': 0.9912314063556458,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecy

In [9]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    99.89%    99.62%    99.65%    99.65%    99.67%
main                              123    99.89%    99.62%    99.66%    99.65%    99.67%
main                              456    99.89%    91.92%    91.73%    94.57%    92.43%
taskB_90rotation                   42    99.67%   100.00%   100.00%   100.00%   100.00%
taskB_label_smoothing_0.0          42    99.67%    99.62%    99.66%    99.65%    99.67%
taskB_label_smoothing_0.2          42    99.45%    99.06%    99.10%    99.11%    99.12%
----------------------------------------------------------------------------------------
main test_acc: 97.06 +/- 3.63%
main test_f1:  97.01 +/- 3.74%
